In [1]:
!pip install git+https://github.com/WildlifeDatasets/wildlife-datasets@develop
!pip install git+https://github.com/WildlifeDatasets/wildlife-tools


  Cloning https://github.com/WildlifeDatasets/wildlife-datasets (to revision develop) to /tmp/pip-req-build-eb6k4k4k
  Running command git clone --filter=blob:none --quiet https://github.com/WildlifeDatasets/wildlife-datasets /tmp/pip-req-build-eb6k4k4k
  Running command git checkout -b develop --track origin/develop
  Switched to a new branch 'develop'
  Branch 'develop' set up to track remote branch 'develop' from 'origin'.
  Resolved https://github.com/WildlifeDatasets/wildlife-datasets to commit e0c7a9e9d8b90719c706f3779d6d231b5b861fb1
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 183.9/183.9 kB 9.7 MB/s eta 0:00:00
  Created wheel for wildlife-datasets: filename=wildlife_datasets-1.0.5-py3-none-any.whl size=88706 sha256=0acdc4c4d2155a785bf6f2904f66107220ac2a407a2b7780a3898299a5d34ff6
  Stored in directory: /tmp/pip-ephem-wheel-cache-65uibv1m/wheels/a

In [2]:
import os
import numpy as np
import pandas as pd
import timm
import torch
import torchvision.models as models
import torchvision.transforms as T
from wildlife_datasets.datasets import AnimalCLEF2025
from wildlife_tools.features import DeepFeatures
from wildlife_tools.similarity import CosineSimilarity
from wildlife_tools.similarity.wildfusion import SimilarityPipeline, WildFusion
from wildlife_tools.similarity.pairwise.lightglue import MatchLightGlue
from wildlife_tools.features.local import AlikedExtractor
from wildlife_tools.similarity.calibration import IsotonicCalibration


def create_sample_submission(dataset_query, predictions, file_name='sample_submission.csv'):
    df = pd.DataFrame({
        'image_id': dataset_query.metadata['image_id'],
        'identity': predictions
    })
    df.to_csv(file_name, index=False)

2025-04-16 06:36:36.420749: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:477] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1744785396.593481      31 cuda_dnn.cc:8310] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1744785396.644400      31 cuda_blas.cc:1418] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered


In [8]:
class EfficientNetFeatures(DeepFeatures):
    def __init__(self, model_name='efficientnet_b0', device='cuda', batch_size=16):
        # 先创建模型
        model = models.efficientnet_b0(pretrained=True)
        # 移除最后的分类层，只保留特征提取部分
        self.model = torch.nn.Sequential(
            model.features,
            model.avgpool
        )
        # 将模型传给父类初始化
        super().__init__(model=self.model)
        
        # 其他设置
        self.model = self.model.to(device)
        self.model.eval()
        self.device = device
        self.batch_size = batch_size

    def extract(self, images):
        """从图像中提取EfficientNet特征"""
        features = []
        with torch.no_grad():
            for i in range(0, len(images), self.batch_size):
                batch = images[i:i+self.batch_size]
                if not isinstance(batch[0], torch.Tensor):
                    batch = torch.stack([img for img in batch])
                batch = batch.to(self.device)
                
                feature = self.model(batch)
                # 将特征展平为一维向量
                feature = torch.flatten(feature, 1)
                features.append(feature.cpu().numpy())
        
        return np.vstack(features)

In [9]:
root = '/kaggle/input/animal-clef-2025'
transform_display = T.Compose([
    T.Resize([384, 384]),
])
transform = T.Compose([
    *transform_display.transforms,
    T.ToTensor(),
    T.Normalize(mean=(0.485, 0.456, 0.406), std=(0.229, 0.224, 0.225))
])

transforms_aliked = T.Compose([
    T.Resize([512, 512]),
    T.ToTensor()
])

# EfficientNet需要的图像变换
transform_efficientnet = T.Compose([
    T.Resize([224, 224]),  # EfficientNet通常使用224x224的输入
    T.ToTensor(),
    T.Normalize(mean=(0.485, 0.456, 0.406), std=(0.229, 0.224, 0.225))
])



In [10]:
# Loading the dataset
dataset = AnimalCLEF2025(root, load_label=True)
dataset_database = dataset.get_subset(dataset.metadata['split'] == 'database')
dataset_query = dataset.get_subset(dataset.metadata['split'] == 'query')
dataset_calibration = AnimalCLEF2025(root, df=dataset_database.metadata[:100], load_label=True) 

n_query = len(dataset_query)



In [6]:
# Loading the models
name = 'hf-hub:BVRA/MegaDescriptor-L-384'
model = timm.create_model(name, num_classes=0, pretrained=True)
device = 'cuda'

matcher_aliked = SimilarityPipeline(
    matcher = MatchLightGlue(features='aliked', device=device, batch_size=16),
    extractor = AlikedExtractor(),
    transform = transforms_aliked,
    calibration = IsotonicCalibration()
)

matcher_mega = SimilarityPipeline(
    matcher = CosineSimilarity(),
    extractor = DeepFeatures(model=model, device=device, batch_size=16),
    transform = transform,
    calibration = IsotonicCalibration()
)


config.json:   0%|          | 0.00/609 [00:00<?, ?B/s]

pytorch_model.bin:   0%|          | 0.00/1.94G [00:00<?, ?B/s]

/usr/local/lib/python3.11/dist-packages/lightglue/lightglue.py:24: FutureWarning: `torch.cuda.amp.custom_fwd(args...)` is deprecated. Please use `torch.amp.custom_fwd(args..., device_type='cuda')` instead.
  @torch.cuda.amp.custom_fwd(cast_inputs=torch.float32)
Downloading: "https://github.com/cvg/LightGlue/releases/download/v0.1_arxiv/aliked_lightglue.pth" to /root/.cache/torch/hub/checkpoints/aliked_lightglue_v0-1_arxiv.pth
100%|██████████| 45.4M/45.4M [00:00<00:00, 50.7MB/s]
Downloading: "https://github.com/Shiaoming/ALIKED/raw/main/models/aliked-n16.pth" to /root/.cache/torch/hub/checkpoints/aliked-n16.pth
100%|██████████| 2.61M/2.61M [00:00<00:00, 276MB/s]


In [14]:
# 创建EfficientNet特征提取管道
matcher_efficientnet = SimilarityPipeline(
    matcher = CosineSimilarity(),
    extractor = EfficientNetFeatures(device=device, batch_size=32),
    transform = transform_efficientnet,
    calibration = IsotonicCalibration()
)


# Calibrating the WildFusion - 现在包含三个管道：ALIKED、MegaDescriptor和EfficientNet
wildfusion = WildFusion(
    calibrated_pipelines = [matcher_aliked, matcher_mega, matcher_efficientnet], 
    priority_pipeline = matcher_mega
)
wildfusion.fit_calibration(dataset_calibration, dataset_calibration)


# Compute WildFusion similarity
similarity = wildfusion(dataset_query, dataset_database, B=25)


pred_idx = similarity.argsort(axis=1)[:,-1]
pred_scores = similarity[range(n_query), pred_idx]


/usr/local/lib/python3.11/dist-packages/torchvision/models/_utils.py:208: UserWarning: The parameter 'pretrained' is deprecated since 0.13 and may be removed in the future, please use 'weights' instead.
  warnings.warn(
/usr/local/lib/python3.11/dist-packages/torchvision/models/_utils.py:223: UserWarning: Arguments other than a weight enum or `None` for 'weights' are deprecated since 0.13 and may be removed in the future. The current behavior is equivalent to passing `weights=EfficientNet_B0_Weights.IMAGENET1K_V1`. You can also use `weights=EfficientNet_B0_Weights.DEFAULT` to get the most up-to-date weights.
  warnings.warn(msg)
100%|█████████████████████████████████████████████████████████████████| 4/4 [00:02<00:00,  1.68it/s]


RuntimeError: Expected size for first two dimensions of batch2 tensor to be: [128000, 1] but got: [128000, 1280].

In [ ]:


new_individual = 'new_individual'
threshold = 0.6
labels = dataset_database.labels_string
predictions = labels[pred_idx]
predictions[pred_scores < threshold] = new_individual
create_sample_submission(dataset_query, predictions, file_name='sample_submission.csv')